In [1]:
%load_ext autoreload
%autoreload 2
%env CUPY_ACCELERATORS=cutensor,cub

env: CUPY_ACCELERATORS=cutensor,cub


In [2]:
import tensorly as tl
import plotly.io as pio
#pio.renderers.default = 'iframe'
tl.set_backend('numpy')
tl.tenalg.set_backend('einsum')
tl.plugins.use_opt_einsum()
print(f'TensorLy backend: {tl.get_backend()}')

TensorLy backend: numpy


In [3]:
from moabb.datasets import *
from moabb.paradigms import P300

dataset = BNCI2014_009()
paradigm = P300()
X, y, meta = paradigm.get_data(dataset)
X = tl.tensor(X)
X.shape

/data/leuven/352/vsc35289/miniconda3/envs/bttda-cuda/lib/python3.11/site-packages/urllib3/connectionpool.py:1064: InsecureRequestWarning: Unverified HTTPS request is being made to host 'lampx.tugraz.at'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(
100%|█████████████████████████████████████| 18.5M/18.5M [00:00<00:00, 30.3GB/s]
SHA256 hash of downloaded file: beddf78f1834ddef15553e32c9d18c46bc9b3fd244ef3a8e2fe362066dfb027d
Use this value as the 'known_hash' argument of 'pooch.retrieve' to ensure that the file hasn't changed if it is downloaded again in the future.
/data/leuven/352/vsc35289/miniconda3/envs/bttda-cuda/lib/python3.11/site-packages/moabb/datasets/preprocessing.py:278: UserWarning: warnEpochs <Epochs | 576 events (all good), 0 – 0.801 s (baseline off), ~14.5 MiB, data loaded,
 'Target': 96
 'NonTarget': 480>
  warn(f"warnEpochs {epochs}")
/data/leuven/352/vsc35289/minicond

(17280, 16, 206)

In [4]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.decomposition import PCA
from hoda.classification import SelectFCutoff
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

clf = make_pipeline(
    FunctionTransformer(tl.to_numpy),
    PCA(n_components=None, whiten=True),
    SelectFCutoff(cutoff=1),
    LinearDiscriminantAnalysis(shrinkage='auto', solver='lsqr')
)

In [5]:
hoda_params=  dict(
    max_iter=64,
    toeplitz=(1,),
    taper=False,
    verbose=False,
    refit_shrinkage=True,
)

In [6]:
from sklearn.model_selection import StratifiedKFold

cv = StratifiedKFold(n_splits=5)

bttdacv_params = dict(
    hoda_params=hoda_params,
    verbose=False,
    cv=cv,
    n_jobs=5*10,
    clf = clf,
)

In [7]:
from hoda.classification import BTTDACV

bttdacv = BTTDACV(
    max_n_blocks=2,
    fixed_n_blocks=True,
    thetas=[0,0.1,0.2,0.3,0.4,0.5,0.6, 0.7, 0.8, 0.9,1],
    **bttdacv_params
)

In [ ]:
bttdacv.fit(X,y)

/vsc-hard-mounts/leuven-data/352/vsc35289/hoda-bci/src/hoda/hoda.py:357: UserWarning: Maximum number of iterations reached without convergence
  warnings.warn("Maximum number of iterations reached without convergence")
/vsc-hard-mounts/leuven-data/352/vsc35289/hoda-bci/src/hoda/hoda.py:357: UserWarning: Maximum number of iterations reached without convergence
  warnings.warn("Maximum number of iterations reached without convergence")
/vsc-hard-mounts/leuven-data/352/vsc35289/hoda-bci/src/hoda/hoda.py:357: UserWarning: Maximum number of iterations reached without convergence
  warnings.warn("Maximum number of iterations reached without convergence")
/vsc-hard-mounts/leuven-data/352/vsc35289/hoda-bci/src/hoda/hoda.py:357: UserWarning: Maximum number of iterations reached without convergence
  warnings.warn("Maximum number of iterations reached without convergence")
/vsc-hard-mounts/leuven-data/352/vsc35289/hoda-bci/src/hoda/hoda.py:357: UserWarning: Maximum number of iterations reached w

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
%matplotlib inline
for b in bttdacv.blocks_:
    print(b.aps_[1].shape)
    plt.plot(b.aps_[1])
    plt.show()